<a href="https://colab.research.google.com/github/Harukokoko/Haruka-Nakagawa/blob/main/%E2%98%85uncomdata_results_HS8482_%E6%95%B0%E5%80%A4%E8%A8%88%E7%AE%97%E3%81%BE%E3%81%A8%E3%82%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Callable
import scipy as sp
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import io
import os
from google.colab import files
from google.colab import drive

In [11]:
# Google Driveをマウント
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/16-research/99_Sotsuron/ex_HS8482_1995_2023.xlsx'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ExcelファイルをPandasデータフレームに読み込む
df = pd.read_excel(file_path)

# データフレームを確認
df.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,20,AND,Andorra,X,...,19.61,False,0.0,False,NaN,5902.458,5902.458,0,False,True
1,C,A,20230101,2023,52,2023,20,AND,Andorra,X,...,2.76,False,0.0,False,NaN,51.865,51.865,0,False,True
2,C,A,20230101,2023,52,2023,20,AND,Andorra,X,...,0.25,False,0.0,False,NaN,17.995,17.995,0,False,True
3,C,A,20230101,2023,52,2023,20,AND,Andorra,X,...,1.00,False,0.0,False,NaN,28.939,28.939,0,False,True
4,C,A,20230101,2023,52,2023,20,AND,Andorra,X,...,15.60,False,0.0,False,NaN,5803.660,5803.660,0,False,True


In [13]:
#　データ数
print("列数：",df.shape[0])
print("行数：",df.shape[1])
print("国数：",len(df['reporterDesc'].unique()))

列数： 197106
行数： 47
国数： 193


In [14]:
df.isnull().sum()

,0
typeCode,0
freqCode,0
refPeriodId,0
refYear,0
refMonth,0
period,0
reporterCode,0
reporterISO,0
reporterDesc,0
flowCode,0


In [15]:
# 欠損値の割合を計算
threshold = 0.5  # 50%以上が欠損している列を削除
df = df.loc[:, df.isna().mean() < threshold]

In [16]:
df = df[df['reporterDesc'] != 'world']
df = df[df['partnerDesc'] != 'world']
# 結果を確認
print(f"Number of rows after filtering: {len(df)}")

Number of rows after filtering: 197106


In [17]:
# 必要な列のみ抽出
df = df[["refYear", "reporterDesc", "partnerDesc", "fobvalue"]]

# FOB値が0以下の場合は除外
df = df[df["fobvalue"] > 0]

# ノード名を小文字に統一
df["reporterDesc"] = df["reporterDesc"].str.lower()
df["partnerDesc"] = df["partnerDesc"].str.lower()

# 年ごとにグラフを作成
graphs_by_year = {}

for year, group in df.groupby("refYear"):
    # 有向グラフを作成
    G = nx.DiGraph()

    # エッジを一括追加
    edges = group[["reporterDesc", "partnerDesc", "fobvalue"]].to_dict(orient="records")
    G.add_edges_from([(edge["reporterDesc"], edge["partnerDesc"], {"weight": edge["fobvalue"]}) for edge in edges])

    # 年ごとのグラフを保存
    graphs_by_year[year] = G

    # 年ごとのグラフ情報を表示
    print(f"Year: {year}")
    print(f"  Number of nodes: {G.number_of_nodes()}")
    print(f"  Number of edges: {G.number_of_edges()}")

# 年ごとのグラフが保存された辞書
print(f"Total years processed: {len(graphs_by_year)}")

Year: 1995
  Number of nodes: 212
  Number of edges: 3190
Year: 1996
  Number of nodes: 211
  Number of edges: 3424
Year: 1997
  Number of nodes: 209
  Number of edges: 3466
Year: 1998
  Number of nodes: 211
  Number of edges: 3666
Year: 1999
  Number of nodes: 213
  Number of edges: 3798
Year: 2000
  Number of nodes: 225
  Number of edges: 4389
Year: 2001
  Number of nodes: 228
  Number of edges: 4626
Year: 2002
  Number of nodes: 225
  Number of edges: 4700
Year: 2003
  Number of nodes: 227
  Number of edges: 4898
Year: 2004
  Number of nodes: 226
  Number of edges: 5156
Year: 2005
  Number of nodes: 230
  Number of edges: 5314
Year: 2006
  Number of nodes: 231
  Number of edges: 5547
Year: 2007
  Number of nodes: 232
  Number of edges: 5767
Year: 2008
  Number of nodes: 228
  Number of edges: 5845
Year: 2009
  Number of nodes: 231
  Number of edges: 5924
Year: 2010
  Number of nodes: 231
  Number of edges: 6132
Year: 2011
  Number of nodes: 232
  Number of edges: 6226
Year: 2012
  N

In [18]:
def calculate_flow_with_stop_on_target(G, source, target, weight='weight'):
    """
    始点から終点までの流入量を計算し、終点に到達したら計算を終了する関数。
    また、終点から始点へのエッジを削除する。

    Parameters:
        G (nx.DiGraph): 重み付き有向グラフ
        source (str): 始点ノード
        target (str): 終点ノード
        weight (str): 重み属性の名前

    Returns:
        float: 終点への最終流入量
    """
    # 終点から始点へのエッジを削除
    if G.has_edge(target, source):
        G = G.copy()  # オリジナルを変更しないためにコピー
        G.remove_edge(target, source)

    # 初期化: 各ノードの流入量を0に設定
    flow = {node: 0 for node in G.nodes}
    flow[source] = 1  # 始点の流入量を1に設定

    visited = set()  # 訪問済みノードを記録

    def dfs(node):
        if node == target:
            return  # 終点に到達したら終了

        visited.add(node)
        out_edges = G.out_edges(node, data=True)
        total_weight = sum(edge_data[weight] for _, _, edge_data in out_edges)

        if total_weight == 0:  # 出力がない場合
            return

        for _, neighbor, edge_data in out_edges:
            proportion = edge_data[weight] / total_weight
            if neighbor not in visited:
                flow[neighbor] += flow[node] * proportion
                dfs(neighbor)  # 再帰的に隣接ノードを処理

    # DFSを開始
    dfs(source)

    return flow.get(target, 0)

In [19]:
# 基点と終点を設定
source_node = "china"
target_node = "usa"

# 年ごとの結果を保存するリスト
yearly_results = []

# 年ごとにグラフを処理
for year, G in graphs_by_year.items():

    # サブグラフ（距離4以下のノードのみを含む）
    paths_within_distance = list(nx.all_simple_paths(G, source=source_node, target=target_node, cutoff=4))
    nodes_within_paths = set(node for path in paths_within_distance for node in path)
    subgraph = G.subgraph(nodes_within_paths)

    # アメリカへの最終流入量を計算
    usa_flow = calculate_flow_with_stop_on_target(subgraph, source=source_node, target=target_node, weight="weight")

    # 直接輸出割合を計算
    if subgraph.has_edge(source_node, target_node):
        direct_weight = subgraph[source_node][target_node]["weight"]
        total_source_weight = sum(edge_data["weight"] for _, _, edge_data in subgraph.out_edges(source_node, data=True))
        direct_export_ratio = direct_weight / total_source_weight
    else:
        direct_export_ratio = 0  # 直接輸出がない場合

    # 直間比率を計算
    if usa_flow > 0:
        The_ratio_of_di = direct_export_ratio / usa_flow
        The_ratio_of_indi = 1 - The_ratio_of_di
        Indi_to_di = The_ratio_of_indi / The_ratio_of_di
    else:
        The_ratio_of_di = 0
        The_ratio_of_indi = 0
        Indi_to_di = 0

    # 中国の全輸出額を計算
    china_total_fobvalue = df[df["reporterDesc"].str.lower() == "china"]["fobvalue"].sum()

    # サブグラフ内の中国の全輸出額
    if subgraph.has_node("china"):
        china_total_export = sum(edge_data["weight"] for _, _, edge_data in subgraph.out_edges("china", data=True))
    else:
        china_total_export = 0

    # サブグラフ内の中国から米国への直接輸出額
    direct_weight = subgraph[source_node][target_node]["weight"] if subgraph.has_edge(source_node, target_node) else 0

    # アメリカへの推定流入額
    usa_final_estimation = usa_flow * china_total_export

    # 年ごとの結果を保存
    yearly_results.append({
        "Year": year,
        "Flow to USA": usa_flow,
        "Direct Export Ratio": direct_export_ratio,
        "Direct Export Amount": direct_weight,
        "Indirect to Direct Ratio": Indi_to_di,
        "China's Total FOB Value": china_total_fobvalue,
        "China's Total Export Volume": china_total_export,
        "Estimated Final Flow to USA": usa_final_estimation
    })

# データフレームに変換
results_df = pd.DataFrame(yearly_results)

# データフレームをCSVとして保存
results_df.to_csv("yearly_export_results.csv", index=False)
print("Yearly export results saved to 'yearly_export_results.csv'.")

# データフレームの表示
print(results_df.head())

Yearly export results saved to 'yearly_export_results.csv'.
   Year  Flow to USA  Direct Export Ratio  Direct Export Amount  \
0  1995     0.283363             0.282416            84602953.0   
1  1996     0.257576             0.255821            89434945.0   
2  1997     0.284257             0.284244           112014302.0   
3  1998     0.327357             0.327346           139911088.0   
4  1999     0.287938             0.287918           139727590.0   

   Indirect to Direct Ratio  China's Total FOB Value  \
0                  0.003351             2.016721e+11   
1                  0.006858             2.016721e+11   
2                  0.000046             2.016721e+11   
3                  0.000031             2.016721e+11   
4                  0.000069             2.016721e+11   

   China's Total Export Volume  Estimated Final Flow to USA  
0                  299568398.0                 8.488650e+07  
1                  349599521.0                 9.004828e+07  
2             

In [20]:
results_df.head()

,Year,Flow to USA,Direct Export Ratio,Direct Export Amount,Indirect to Direct Ratio,China's Total FOB Value,China's Total Export Volume,Estimated Final Flow to USA
0,1995,0.283363,0.282416,84602953.0,0.003351,2.016721e+11,299568398.0,8.488650e+07
1,1996,0.257576,0.255821,89434945.0,0.006858,2.016721e+11,349599521.0,9.004828e+07
2,1997,0.284257,0.284244,112014302.0,0.000046,2.016721e+11,394078098.0,1.120195e+08
3,1998,0.327357,0.327346,139911088.0,0.000031,2.016721e+11,427409873.0,1.399155e+08
4,1999,0.287938,0.287918,139727590.0,0.000069,2.016721e+11,485302732.0,1.397372e+08


In [21]:
import pandas as pd
import networkx as nx

# 前年比変化率を計算して追加
results_df["DiIndiRatio Change (%)"] = results_df["Indirect to Direct Ratio"].pct_change() * 100

# 変化率の絶対値が大きい年を特定（例: 変化率が50%以上の年）
threshold = 50  # しきい値を設定
significant_years = results_df[results_df["DiIndiRatio Change (%)"].abs() > threshold]["Year"].tolist()

# 結果を保存するリスト
significant_results = []

# 流量変化が大きな経路を抽出
for year in significant_years:
    # 現在の年と前の年のグラフを取得
    current_graph = graphs_by_year.get(year)
    previous_graph = graphs_by_year.get(year - 1)

    if not previous_graph:
        continue  # 前年のグラフがない場合スキップ

    # 流量（エッジ重みの合計）を計算する関数
    def calculate_edge_weights(G, source, target, cutoff=4):
        """
        指定された始点から終点までのエッジの流量を計算する関数。
        """
        edge_weights = {}
        paths = list(nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff))
        for path in paths:
            total_weight = sum(G[path[i]][path[i + 1]].get("weight", 0) for i in range(len(path) - 1))
            edge_weights[tuple(path)] = total_weight
        return edge_weights

    # 現在の年と前の年の流量を計算
    current_edge_weights = calculate_edge_weights(current_graph, "china", "usa", cutoff=4)
    previous_edge_weights = calculate_edge_weights(previous_graph, "china", "usa", cutoff=4)

    # 流量の変化率を計算
    edge_weight_changes = {}
    all_paths = set(current_edge_weights.keys()).union(previous_edge_weights.keys())
    for path in all_paths:
        current_weight = current_edge_weights.get(path, 0)
        previous_weight = previous_edge_weights.get(path, 0)
        if previous_weight > 0:  # 前年の流量が0以上の場合に変化率を計算
            change_rate = ((current_weight - previous_weight) / previous_weight) * 100
        else:
            change_rate = 100 if current_weight > 0 else 0  # 無限大を避ける
        edge_weight_changes[path] = change_rate

    # 流量変化率の上位10経路を抽出
    top_10_changes = sorted(
        edge_weight_changes.items(), key=lambda x: abs(x[1]), reverse=True
    )[:10]

    # 保存
    significant_results.append({
        "Year": year,
        "DiIndiRatio Change (%)": results_df.loc[results_df["Year"] == year, "DiIndiRatio Change (%)"].values[0],
        "Top 10 Path Changes": {path: change for path, change in top_10_changes}
    })

# 最終結果を年ごとに表示
print("\nSignificant DiIndiRatio Change (%) (Top 10 Path Flow Changes):")
for result in significant_results:
    print(f"\nYear: {result['Year']} (DiIndiRatio Change (%): {result['DiIndiRatio Change (%)']:.2f}%)")
    print("Top 10 Path Flow Changes:")
    if result["Top 10 Path Changes"]:
        for path, change in result["Top 10 Path Changes"].items():
            print(f"  Path: {' -> '.join(path)}, Change Rate: {change:.2f}%")
    else:
        print("  No significant paths.")


Significant DiIndiRatio Change (%) (Top 10 Path Flow Changes):

Year: 1996 (DiIndiRatio Change (%): 104.62%)
Top 10 Path Flow Changes:
  Path: china -> czechia -> china, hong kong sar -> ireland -> usa, Change Rate: 5352.21%
  Path: china -> czechia -> ecuador -> usa, Change Rate: 2148.22%
  Path: china -> czechia -> finland -> portugal -> usa, Change Rate: 1819.83%
  Path: china -> czechia -> ireland -> usa, Change Rate: 1742.23%
  Path: china -> mexico -> belize -> usa, Change Rate: 1677.13%
  Path: china -> czechia -> mexico -> belize -> usa, Change Rate: 1663.66%
  Path: china -> mexico -> china, hong kong sar -> ireland -> usa, Change Rate: 1512.07%
  Path: china -> czechia -> denmark -> portugal -> usa, Change Rate: 1421.23%
  Path: china -> mexico -> indonesia -> usa, Change Rate: 1360.78%
  Path: china -> czechia -> mexico -> indonesia -> usa, Change Rate: 1345.24%

Year: 1997 (DiIndiRatio Change (%): -99.33%)
Top 10 Path Flow Changes:
  Path: china -> norway -> rep. of korea 